In [ ]:
import numpy as np
import tensorflow as tf
from scipy.linalg import expm
from scipy.optimize import minimize
from qugradlab.systems.semiconducting.esr import SpinChain
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pulse_vqe_entanglement.dataclasses import SimulationConfig

In [77]:
def _initialise_x0(config: SimulationConfig) -> np.ndarray:
    """Initial control parameters drawn uniformly in [-1, 1]."""
    n = config.n_time_steps * config.n_J_channels
    x0 = config.J_init_scale * np.random.uniform(-1.0, 1.0, size=n) + config.J_init_mean
    return np.clip(x0, -1.0, 1.0)

def convert_dtype(np_array) -> tf.Tensor:
    """Convert numpy array to TensorFlow complex128 tensor."""
    return tf.cast(tf.convert_to_tensor(np_array), dtype=tf.complex128)


def single_qubit_excitation(theta: float) -> np.ndarray:
    """
    Single-qubit rotation gate in the two-spin subspace.

    Rotates the |01⟩ ↔ |10⟩ subspace by angle theta while leaving |00⟩ and
    |11⟩ unchanged. Basis order: [|00⟩, |01⟩, |10⟩, |11⟩].
    """
    return np.array(
        [[1, 0, 0, 0],
         [0, np.cos(theta), -np.sin(theta), 0],
         [0, np.sin(theta),  np.cos(theta), 0],
         [0, 0, 0, 1]],
        dtype=np.complex128,
    )

def _cost_entropy(driven_sys, x, *, S_max: float, ent_traj_fn) -> float:
    """
    Mean squared trajectory violation of the entanglement cap.

    cost = mean((E(t) - S_max)₊²) where E is computed by ent_traj_fn.

    Parameters
    ----------
    ent_traj_fn : callable
        A traj_fn from STATE_MEASURES: takes (4, n_t) trajectory, returns (n_t,).
    """
    traj = np.asarray(driven_sys.propagate_all(x))
    viol = np.maximum(0.0, ent_traj_fn(traj) - S_max)
    return float(np.mean(viol**2))


def _entropy_grad_fd(driven_sys, x, *, S_max: float, eps: float,
                     ent_traj_fn, f0: float | None = None) -> np.ndarray:
    """
    Forward-difference gradient of the entropy cost.

    Parameters
    ----------
    ent_traj_fn : callable
        Trajectory entanglement function; see _cost_entropy.
    f0 : float, optional
        Cost at the unperturbed x. Providing it avoids a redundant evaluation.
    """
    x = np.asarray(x, float)
    if f0 is None:
        f0 = _cost_entropy(driven_sys, x, S_max=S_max, ent_traj_fn=ent_traj_fn)
    g = np.empty_like(x)
    for i in range(len(x)):
        xp = x.copy()
        xp[i] += eps
        g[i] = (_cost_entropy(driven_sys, xp, S_max=S_max, ent_traj_fn=ent_traj_fn) - f0) / eps
    return g


def _lagrangian_and_grad(driven_sys, x, *, observable, lam, S_max, fd_eps,
                         rho, eps_infidelity, ent_traj_fn):
    """
    Augmented Lagrangian value and gradient.

    Objective: L = S_cost + λ(I - ε) + (ρ/2)(I - ε)²

    where S_cost is the mean-squared entanglement violation (using ent_traj_fn)
    and I is the state infidelity (the constraint I ≤ ε is enforced via λ).
    """
    S = _cost_entropy(driven_sys, x, S_max=S_max, ent_traj_fn=ent_traj_fn)
    gS = _entropy_grad_fd(driven_sys, x, S_max=S_max, eps=fd_eps, ent_traj_fn=ent_traj_fn, f0=S)

    infidelity, grad_inf = driven_sys.gradient(x, observable)
    infidelity = max(0.0, float(np.real(infidelity)))   # guard against float64 cancellation
    grad_inf = np.asarray(grad_inf, float)
    c = infidelity - eps_infidelity

    L = S + lam * c + 0.5 * rho * c**2
    gL = gS + (lam + rho * c) * grad_inf
    return float(L), gL


def _primal_dual_optimise(driven_sys, x0, *, observable, bounds, config: SimulationConfig,
                          ent_traj_fn, verbose: bool = True):
    """
    Augmented Lagrangian primal-dual loop.

    Each outer iteration minimises the Lagrangian via L-BFGS-B, then updates
    the dual variable λ ← max(0, λ + α(I - ε)).

    Parameters
    ----------
    ent_traj_fn : callable
        Trajectory entanglement function resolved from config.state_ent_measure.

    Returns
    -------
    x_final : np.ndarray
    lam_final : float
    history : list[dict]  – per-outer-iteration diagnostics
    """
    x = np.asarray(x0, float).copy()
    lam = 1.0
    hist = []

    for k in range(int(config.n_outer)):
        res = minimize(
            lambda z: _lagrangian_and_grad(
                driven_sys, z, observable=observable, lam=lam,
                S_max=float(config.S_max), fd_eps=float(config.fd_eps),
                rho=float(config.rho), eps_infidelity=float(config.eps_infidelity),
                ent_traj_fn=ent_traj_fn,
            ),
            x0=x, jac=True, method="L-BFGS-B", bounds=bounds,
            options={"maxiter": int(config.inner_maxiter), "gtol": float(config.gtol)},
        )
        x = np.asarray(res.x, float)

        S = _cost_entropy(driven_sys, x, S_max=float(config.S_max), ent_traj_fn=ent_traj_fn)

        infidelity = np.real(driven_sys.gradient(x, observable)[0])
        c = infidelity - float(config.eps_infidelity)
        lam = max(0.0, lam + float(config.dual_lr) * c)

        hist.append({
            "outer_iter": k, "lam": lam, "I_final": infidelity, "constraint_residual": c, "S": S,
            "inner_success": bool(res.success), "inner_nit": int(res.nit),
            "inner_fun": float(res.fun),
            "inner_grad_norm": float(np.linalg.norm(res.jac)) if res.jac is not None else np.nan,
            "message": str(res.message),
        })
        if verbose:
            print(f"[outer {k:02d}] I={infidelity:.3e}, lam={lam:.6e}, "
                  f"inner_nit={res.nit}, inner_success={res.success}, "
                  f"constraint_residual={c:.3e}, S={S:.3e}")

    return x, lam, hist

def von_neumann_entropy_single(state: np.ndarray) -> float:
    """
    Von Neumann entropy S(ρ_A) in bits for a 2-qubit pure state.

    Computes rho_A = Tr_B(|ψ⟩⟨ψ|) then S = -Tr(ρ_A log₂ ρ_A).
    """
    rho_A = state.reshape(2, 2) @ state.reshape(2, 2).conj().T
    eig = np.clip(np.linalg.eigvalsh(rho_A), 1e-15, 1.0)
    return float(-np.sum(eig * np.log2(eig)))


def von_neumann_entropy_traj(traj: np.ndarray) -> np.ndarray:
    """
    Von Neumann entropy S(ρ_A) in bits along a 2-qubit state trajectory.

    Parameters
    ----------
    traj : ndarray, shape (4, n_t)
        State at each time step.

    Returns
    -------
    ndarray, shape (n_t,)
    """
    psi = np.asarray(traj).T.reshape(-1, 2, 2)          # (n_t, 2, 2)
    rho_A = psi @ np.conj(psi).transpose(0, 2, 1)        # (n_t, 2, 2)
    eig = np.clip(np.linalg.eigvalsh(rho_A), 1e-15, 1.0)
    return -np.sum(eig * np.log2(eig), axis=1)

### First set config and optimise pulses without noise

Note:
1. Infidelity now must be the density matrix infidelity with noise
2. Need to change plotting functions respectively

In [69]:
config = SimulationConfig(
    zeeman_splittings=np.array([1000.0, 1008.0]),
    initial_state=np.array([0.0, 1.0, 0.0, 0.0], dtype=np.complex128),
    theta_target=0.01 * np.pi,
    n_time_steps=30,
    n_discretisation=1,
    T=300.0,
    max_drive_strength=0.0,
    J_max=10.0,
    J_min=0.0,
    J_init_scale=1e0,
    J_init_mean=0.0,
    feromagnetic=False,
    gtol=1e-12,
    maxiter=500,
    entanglement_optimisation=True,
    state_ent_measure="von_neumann",
    #extra_ent_measures=["linear"],
    J_init_seed=None,

    noise_type="depolarising",
    noise_rate=0.0,

    # primal-dual settings
    S_max=0.01,
    eps_infidelity=1e-12,
    rho=0.0,
    dual_lr=1e10,
    n_outer=5,
    inner_maxiter=500,
    fd_eps=1e-6,
)

In [71]:
NS_TO_US = 1e-3

I2 = np.eye(2, dtype=np.complex128)
X = np.array([[0, 1], [1, 0]], dtype=np.complex128)
Y = np.array([[0, -1j], [1j, 0]], dtype=np.complex128)
Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)
Z1 = np.kron(Z, I2)
Z2 = np.kron(I2, Z)

B = np.array(config.zeeman_splittings, dtype=np.float64)
B_avg = np.mean(B)
deltaB = B - B_avg
U_rot = expm(-1j * np.pi 
                * (deltaB[0] * Z1 + deltaB[1] * Z2) 
                * config.T * NS_TO_US)

target_state = U_rot @ single_qubit_excitation(config.theta_target) @ config.initial_state
observable = np.eye(4, dtype=np.complex128) - np.outer(target_state, target_state.conj())

sys = SpinChain(
    spins=2,
    zeeman_splittings=deltaB,
    max_drive_strength=config.max_drive_strength,
    J_max=config.J_max,
    J_min=config.J_min,
    feromagnetic=config.feromagnetic,
    use_graph=True,
)

In [72]:
n_steps = config.n_time_steps
n_disc = config.n_discretisation
n_fine = n_steps * n_disc
dt_fine_us = (config.T / n_fine) * NS_TO_US

def pulse_function(x):
    x_coarse = tf.reshape(convert_dtype(x), (n_steps, config.n_J_channels))
    x_fine = tf.repeat(x_coarse, repeats=n_disc, axis=0)
    return (
        convert_dtype(np.zeros((n_fine, 2))),       # drive amplitudes (disabled)
        convert_dtype(config.zeeman_splittings),    # drive frequencies
        x_fine,                                     # J coupling amplitudes
        convert_dtype(config.initial_state),        # initial state
        dt_fine_us * 2 * np.pi,                     # timestep (rad/μs)
    )

driven_sys = sys.pulse_form(pulse_function)

In [73]:
x0 = _initialise_x0(config)
bounds = [(-1.0, 1.0)] * len(x0)

In [74]:
res = minimize(
    driven_sys.gradient, x0, args=(observable,),
    jac=True, bounds=bounds, method="L-BFGS-B",
    options={"maxiter": int(config.maxiter), "gtol": float(config.gtol)},
)
gn = float(np.linalg.norm(res.jac)) if res.jac is not None else 0.0
S = _cost_entropy(driven_sys, res.x, S_max=float(config.S_max), ent_traj_fn=von_neumann_entropy_traj)
print(f"Infidelity: I={res.fun:.3e}, nit={res.nit}, success={res.success}, grad_norm={gn:.3e}, S={S:.3e}")

xf = res.x
print(xf)

Tracing control amplitude graph.
Infidelity: I=1.868e-11, nit=5, success=True, grad_norm=4.563e-06, S=2.192e-01
[-0.54632628 -0.77043962 -0.6571897   0.24703225  0.39211041 -0.92350396
 -0.2319307  -0.55296198  0.99999492  0.57926035  0.87063044  0.13039016
  0.74092725  0.01478868 -0.19212062 -0.97480236  0.55792578  0.31166851
 -0.24830067  0.37846647 -0.4452549  -0.66811875 -0.65243338  0.4814633
 -0.99999426  0.2203496  -0.0269937  -0.93934101 -0.5978253   0.09399565]


In [75]:
xf_ent, lam, hist = _primal_dual_optimise(
            driven_sys, x0, observable=observable, bounds=bounds,
            config=config, ent_traj_fn=von_neumann_entropy_traj, verbose=True,
        )

print(xf_ent)

[outer 00] I=7.231e-10, lam=8.220878e+00, inner_nit=32, inner_success=True, constraint_residual=7.221e-10, S=5.102e-08
[outer 01] I=9.782e-12, lam=8.308696e+00, inner_nit=2, inner_success=True, constraint_residual=8.782e-12, S=5.223e-08
[outer 02] I=9.574e-12, lam=8.394438e+00, inner_nit=1, inner_success=True, constraint_residual=8.574e-12, S=5.224e-08
[outer 03] I=9.381e-12, lam=8.478247e+00, inner_nit=1, inner_success=True, constraint_residual=8.381e-12, S=5.224e-08
[outer 04] I=9.197e-12, lam=8.560213e+00, inner_nit=1, inner_success=True, constraint_residual=8.197e-12, S=5.224e-08
[-0.92310701 -0.99336187 -0.99848594 -0.99631395 -0.99296727 -0.99985405
 -0.86934763 -0.98734554 -0.94129305 -0.98804631 -0.9818466  -0.92332762
 -0.97674225 -0.97346914 -0.97017136 -0.99780846 -0.85723856 -0.99901425
 -1.         -0.85666704 -0.94008312 -0.99967553 -0.99907023 -0.8141699
 -0.99989092 -0.99998082 -0.74888202 -0.98809884 -0.9120176  -0.88527993]


### Now add Lindblad dynamics

In [76]:
def local_depolarising_ops_2q(gamma):
    pref = np.sqrt(gamma / 4.0)

    return np.array([
        pref * np.kron(X, I2),
        pref * np.kron(Y, I2),
        pref * np.kron(Z, I2),
        pref * np.kron(I2, X),
        pref * np.kron(I2, Y),
        pref * np.kron(I2, Z),
    ], dtype=np.complex128)

L0 = local_depolarising_ops_2q(config.noise_rate)

driven_sys_noisy = driven_sys.add_lindblad_terms(L0=L0)

final_state_noisy = driven_sys_noisy.propagate(xf)
print(final_state_noisy)

Tracing control amplitude graph.
[[-1.49941309e-15-2.88058605e-15j  9.77849653e-01-2.06938363e-01j
  -2.10410925e-02+2.33204284e-02j  1.28599241e-15-3.69978143e-16j]
 [ 1.77175885e-02+5.06079302e-02j  3.25672188e-02-2.49800022e-15j
  -8.36096962e-02-9.95966262e-01j  9.77849653e-01+2.06938363e-01j]
 [-4.90240252e-01+1.63432034e+00j -8.36096962e-02+9.95966262e-01j
  -3.25672188e-02-3.33068491e-16j -2.10410925e-02-2.33204284e-02j]
 [ 3.76997304e-16+8.79022760e-16j  7.70911290e-01-1.18478802e+00j
   2.27933586e-03+4.43615209e-02j -3.32057542e-16-5.06240338e-16j]]


E0000 00:00:1777973537.435636 8465902 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Subshape end out of bounds: -2, for shape with rank 1
